In [1]:

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_DIR = "./newslens_ner_model"

# Load the trained model and tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    use_fast=True
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_DIR
).to(device)

model.eval()

print("Model loaded successfully!")
print("Device:", device)
print("Number of labels:", model.config.num_labels)
print("Maximum model input length:", model.config.max_position_embeddings)


W1018 22:23:13.789000 34660 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda
Number of labels: 37
Maximum model input length: 512


In [4]:

def extract_entities(text, max_length=256, stride=64):
    """Extract entities with offsets into the original input text."""
    if not isinstance(text, str) or not text.strip():
        return []

    max_length = min(max_length, model.config.max_position_embeddings)
    if stride >= max_length:
        raise ValueError("stride must be smaller than max_length")

    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False,
    )

    # Keep one best prediction for each original word span.
    word_predictions = {}

    for chunk_idx in range(len(encoded["input_ids"])):
        input_ids = torch.tensor(
            [encoded["input_ids"][chunk_idx]], device=device
        )
        attention_mask = torch.tensor(
            [encoded["attention_mask"][chunk_idx]], device=device
        )
        offsets = encoded["offset_mapping"][chunk_idx]
        word_ids = encoded.word_ids(batch_index=chunk_idx)

        with torch.inference_mode():
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits[0]
            probs = torch.softmax(logits, dim=-1)
            pred_ids = logits.argmax(dim=-1).tolist()

        # Group all subwords belonging to each word.
        groups = []
        current_group = []

        for i, word_id in enumerate(word_ids):
            start, end = offsets[i]

            if word_id is None or start == end:
                if current_group:
                    groups.append(current_group)
                    current_group = []
                continue

            if current_group and word_id != current_group[-1]["word_id"]:
                groups.append(current_group)
                current_group = []

            current_group.append({
                "word_id": word_id,
                "token_idx": i,
                "start": start,
                "end": end
            })

        if current_group:
            groups.append(current_group)

        for group in groups:
            first = group[0]
            last = group[-1]

            start = first["start"]
            end = last["end"]
            first_token_idx = first["token_idx"]

            pred_id = pred_ids[first_token_idx]
            tag = id2label.get(pred_id, "O")
            confidence = float(probs[first_token_idx, pred_id].item())

            if tag != "O" and "-" in tag:
                prefix, entity_type = tag.split("-", 1)
            else:
                prefix, entity_type = "O", "O"

            key = (start, end)
            candidate = {
                "start": start,
                "end": end,
                "prefix": prefix,
                "label": entity_type,
                "confidence": confidence
            }

            # Overlapping chunks may predict the same word.
            if (
                key not in word_predictions
                or confidence > word_predictions[key]["confidence"]
            ):
                word_predictions[key] = candidate

    # Decode the merged word-level sequence into entity spans.
    words = sorted(
        word_predictions.values(),
        key=lambda item: (item["start"], item["end"])
    )

    entities = []
    active = None

    def finish_entity():
        nonlocal active
        if active is not None:
            start, end = active["start"], active["end"]
            entities.append({
                "text": text[start:end],
                "label": active["label"],
                "start": start,
                "end": end,
                "confidence": round(
                    sum(active["scores"]) / len(active["scores"]), 4
                )
            })
            active = None

    for word in words:
        prefix = word["prefix"]
        label = word["label"]

        if prefix == "O":
            finish_entity()
            continue

        if prefix == "B" or (
            active is not None and active["label"] != label
        ):
            finish_entity()
            active = {
                "label": label,
                "start": word["start"],
                "end": word["end"],
                "scores": [word["confidence"]]
            }

        elif prefix == "I" and active is None:
            # Repair an orphan I-tag.
            active = {
                "label": label,
                "start": word["start"],
                "end": word["end"],
                "scores": [word["confidence"]]
            }

        else:
            active["end"] = word["end"]
            active["scores"].append(word["confidence"])

    finish_entity()
    return entities


print("Updated inference function ready!")


Updated inference function ready!


In [6]:

sample_article = """
The Indian Space Research Organisation announced on Monday
that its scientists in Bengaluru are preparing a new satellite
mission. ISRO officials said the project will launch in 2027.
The announcement was made by Chairman V. Narayanan.
"""


entities = extract_entities(sample_article)

for entity in entities:
    print(entity)

print("\nTotal entities:", len(entities))



{'text': 'The Indian Space Research Organisation', 'label': 'ORG', 'start': 1, 'end': 39, 'confidence': 0.9876}
{'text': 'Monday', 'label': 'DATE', 'start': 53, 'end': 59, 'confidence': 0.9972}
{'text': 'Bengaluru', 'label': 'GPE', 'start': 83, 'end': 92, 'confidence': 0.9978}
{'text': 'ISRO', 'label': 'ORG', 'start': 132, 'end': 136, 'confidence': 0.9898}
{'text': '2027', 'label': 'DATE', 'start': 179, 'end': 183, 'confidence': 0.9851}
{'text': 'V. Narayanan', 'label': 'PERSON', 'start': 223, 'end': 235, 'confidence': 0.9954}

Total entities: 6


In [8]:

sample_article_2 = """
NASA and SpaceX announced a new collaboration in Washington
on Tuesday. Elon Musk said the next mission could launch
from Florida in March 2027. The European Space Agency will
also participate in the programme.
"""

entities_2 = extract_entities(sample_article_2)

for entity in entities_2:
    print(
        f"{entity['text']:<35} "
        f"{entity['label']:<12} "
        f"{entity['confidence']:.3f}"
    )


NASA                                ORG          0.999
SpaceX                              ORG          0.999
Washington                          GPE          0.999
Tuesday                             DATE         0.997
Elon Musk                           PERSON       0.998
Florida                             GPE          0.999
March 2027                          DATE         0.996
The European Space Agency           ORG          0.999


In [9]:

sample_article_2 = """
The 47th Asian Conference on Remote Sensing (ACRS 2026) concluded its five-day event held from October 12–16, 2026, at the Hotel Taj Vivanta in Dwarka, New Delhi, India.
Organized by the AARS, ISRS, and ISG, the event was heavily supported by the Indian Space Research Organisation (ISRO), NRSC, and SAC.

Key Highlights of ACRS 2026

• Core Theme: "Remote Sensing for Future Earth: Innovation, Inclusion & Impact," focusing on sustainability and disaster resilience.
• Technological Focus: Emphasis on GeoAI, big data, digital twins, and cloud computing.
• Participation & Contributions: Nearly 800 experts from 30+ countries attended, featuring major space agencies like ISRO, JAXA, KARI, NASA, and ESA, alongside academic showcases for the NISAR mission.
• Follow-up Training: The ISPRS Summer School on "GeoAI for Future Earth" runs from October 17–21, 2026
"""

entities_2 = extract_entities(sample_article_2)

for entity in entities_2:
    print(
        f"{entity['text']:<35} "
        f"{entity['label']:<12} "
        f"{entity['confidence']:.3f}"
    )


47th                                ORDINAL      0.983
Asian Conference on Remote Sensing ( EVENT        0.907
ACRS 2026                           EVENT        0.870
five-day                            DATE         0.996
October 12–16, 2026                 DATE         0.996
the Hotel Taj Vivanta               FAC          0.982
Dwarka                              GPE          0.987
New Delhi                           GPE          0.976
India                               GPE          0.826
AARS                                ORG          0.874
ISRS                                ORG          0.961
ISG                                 ORG          0.959
the Indian Space Research Organisation ( ORG          0.961
ISRO)                               ORG          0.820
NRSC                                ORG          0.995
SAC                                 ORG          0.986
ACRS 2026                           EVENT        0.330
Remote Sensing for Future Earth: Innovation, Inclusion & Im